#### Set environment variables in [.env](.env) for LLM API calling

### Import Dependencies

In [ ]:
import sys
sys.path.insert(0, "../")
import promptwizard
import random
from promptwizard.glue.promptopt.instantiate import GluePromptOpt
from promptwizard.glue.promptopt.techniques.common_logic import DatasetSpecificProcessing
from promptwizard.glue.common.utils.file import save_jsonlist
from typing import Any
from tqdm import tqdm
import json
import os
from azure.identity import get_bearer_token_provider, AzureCliCredential
from openai import AzureOpenAI
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv(override = True)


### Below code can be used for LLM-as-a-judge eval

In [ ]:
# extract_between / llm_eval / BBH now live in bbh_processor.py, shared across
# every optimizer notebook (PromptWizard, APE, ProTeGi, MPIR) to keep the
# exact-match default and LLM-judge retarget in one place.
from bbh_processor import BBH, LLM_AS_JUDGE_EVAL

### Create a dataset specific class and define the required functions 

In [ ]:
bbh_processor = BBH()

### Load and save the dataset . 
Set the ```dataset_to_run``` variable to choose 1 among the 19 datasets of BBII to run the optimization on

In [ ]:
# Step 1: Prepare directories
if not os.path.exists("data"):
    os.mkdir("data")

# Must match the dataset_to_run/seed used in promptwizard.ipynb (or whichever
# upstream optimizer notebook produced the pkl this notebook loads below) --
# the pkl filename is now scoped by both, so a mismatch raises
# FileNotFoundError instead of silently refining a different task's prompt.
dataset_to_run = 'reasoning_about_colored_objects'
seed = 42

# Shared, idempotent three-way split (data_prep.py) -- every optimizer
# notebook for this (task, seed) gets the identical train/mpir-validation/test
# partition, which is required for the example-level paired analysis in
# REBUILD.md §3.2 to compare like with like.
from data_prep import prepare_bbh_task_split

split_paths = prepare_bbh_task_split(dataset_to_run, bbh_processor, seed=seed)


### Set paths

In [ ]:
train_file_name = split_paths.train_file_name
val_file_name = split_paths.val_file_name
test_file_name = split_paths.test_file_name
path_to_config = "configs"
llm_config_path = os.path.join(path_to_config, "heuristic/llm_config.yaml")
promptopt_config_path = os.path.join(path_to_config, "heuristic/promptopt_config.yaml")
setup_config_path = os.path.join(path_to_config, "heuristic/setup_config.yaml")

### Create an object for calling prompt optimization and inference functionalities

In [ ]:
gp = GluePromptOpt(promptopt_config_path,
                   setup_config_path,
                   train_file_name,
                   bbh_processor,
                   validation_dataset_jsonl=val_file_name,
                   seed=seed)

### Load PromptWizard Prompt

In [ ]:
import pickle

# Constructed from this notebook's own dataset_to_run/seed, matching how
# promptwizard.ipynb names the file it saves -- if the two notebooks were run
# for different tasks or seeds, this raises FileNotFoundError instead of
# silently loading and refining a different task's prompt (the bug this
# scoping was added to prevent: an earlier version of this notebook loaded a
# fixed "results/promptwizard.pkl" regardless of dataset_to_run).
pkl_path = f"results/promptwizard_{dataset_to_run}_seed{seed}.pkl"
with open(pkl_path, "rb") as f:
    promptwizard_prompt = pickle.load(f)

print(f"promptwizard_prompt (loaded from {pkl_path}):", promptwizard_prompt)


### Use MPIR to embed heuristics into PromptWizard.

In [ ]:
MPIR_prompt = gp.improve_prompt(promptwizard_prompt)


In [ ]:
MPIR_prompt
print(MPIR_prompt)

### Evaluate the optimized prompt

In [ ]:
gp.BEST_PROMPT = MPIR_prompt

# Function call to evaluate the prompt. task_name/condition_name/seed feed the
# tracked results/predictions/<task>_<condition>_<seed>.jsonl file
# (REBUILD.md §5) -- update condition_name to match whichever optimizer this
# MPIR pass was applied on top of (e.g. "promptwizard_mpir", "ape_mpir").
accuracy = gp.evaluate(test_file_name, task_name=dataset_to_run,
                       condition_name="promptwizard_mpir", seed=seed)
